In [2]:
# %% [markdown]
# # # LSTM Model: Walk-Forward Validation for Bitcoin (t+10 Horizon)
# # # Python version 3.11+

# %% [markdown]
# ## 1. Import Libraries
# (Keep Imports as before)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time
import os

import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score

import tensorflow
print(f"Using TensorFlow version: {tensorflow.__version__}")
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks
# Ensure keras-tuner is installed
try:
    import keras_tuner as kt
except ImportError:
    print("Keras Tuner not found. Please install it: pip install keras-tuner -U")
    exit()


import plotly.graph_objects as go
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# %% [markdown]
# ## 2. Configuration

# %%
# --- Defined Parameters ---
ticker = "BTC-USD"
start_date = "2017-11-09"
end_date = "2025-01-01"

# >>> New Parameter: Forecast Horizon <<<
forecast_horizon = 10 # Predict 10 days ahead
print(f"Setting Walk-Forward Horizon to: h = {forecast_horizon}")


# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80
# Define Validation Split Ratio *within* the initial training data for tuning
validation_split_ratio_for_tuning = 0.20 # 20% of initial train data for validation

# LSTM Network Parameters
look_back = 60 # Number of previous days to use for prediction

# Keras Tuner Configuration
MAX_TRIALS = 10 # Number of hyperparameter combinations to try
EXECUTIONS_PER_TRIAL = 2 # Number of models to train per trial
TUNER_EPOCHS = 50 # Max epochs during tuning search
TUNER_PATIENCE = 5 # Early stopping patience during tuning

# Final Training Configuration
FINAL_TRAINING_EPOCHS = 100 # Max epochs for the final model training
FINAL_TRAINING_PATIENCE = 10 # Early stopping patience for final training

# Walk-Forward Configuration
RETRAIN_FREQUENCY = 0 # How often to retrain (0 means train once initially)
RETRAIN_EPOCHS = 5 # Number of epochs for periodic retraining
FINAL_TRAINING_BATCH_SIZE = 32 # Default batch size if not tuned

# Keras Tuner Directory (specific for t+10)
# >>> Modified for t+10 <<<
tuner_dir = 'keras_tuner_lstm_wf_t10'
project_name = f'{ticker}_lstm_wf_tuning_t{forecast_horizon}'

# --- Optional: Clean tuner directory before starting ---
# import shutil
# if os.path.exists(tuner_dir):
#     try:
#         shutil.rmtree(tuner_dir)
#         print(f"Removed existing tuner directory: {tuner_dir}")
#     except OSError as e:
#         print(f"Error removing directory {tuner_dir}: {e}")
# ---------------------------------------------------------

# %% [markdown]
# ## 3. Data Loading and Preparation
# (Keep Section 3 as before)
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True); df_full.dropna(inplace=True)
    if df_full.empty: raise ValueError(f"Data became empty after processing.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min().strftime('%Y-%m-%d')} to {df_full.index.max().strftime('%Y-%m-%d')}.")
except Exception as e:
    raise ValueError(f"Failed to load data for {ticker}: {e}")

# %% [markdown]
# ## 4. Data Splitting (Adjusted for h-step Evaluation)
# (Keep Section 4 as before, using forecast_horizon=10)
n_total = len(df_full)
n_train_val = int(train_split_ratio * n_total)
n_test = n_total - n_train_val - forecast_horizon + 1 # Adjusted for h=10

if n_test <= 0:
     raise ValueError(f"Not enough data for walk-forward with h={forecast_horizon}. Need at least {n_train_val + forecast_horizon} total points.")

train_val_data_df = df_full[:n_train_val]
test_data_full = df_full[n_train_val:] # Holds all data from start of test period

print(f"\nInitial Train+Validation Data: {n_train_val} points ({train_val_data_df.index.min().strftime('%Y-%m-%d')} to {train_val_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data Available (for updates & targets): {len(test_data_full)} points")
print(f"Number of walk-forward steps (predictions to generate & evaluate): {n_test}")
if forecast_horizon > 0 and len(test_data_full) >= forecast_horizon:
    eval_start_date = test_data_full.index[forecast_horizon-1].strftime('%Y-%m-%d')
    eval_end_date = test_data_full.index[-1].strftime('%Y-%m-%d')
    print(f"Evaluation Period Start (target t+{forecast_horizon}): {eval_start_date}")
    print(f"Evaluation Period End (target t+{forecast_horizon}): {eval_end_date}")
else:
    print("Evaluation Period cannot be determined due to insufficient test data for horizon.")


# %% [markdown]
# ## 5. Scaling (Fit on Initial Train Portion Only)
# (Keep Section 5 as before)
print("\n--- Scaling Data ---")
n_val_tune = int(validation_split_ratio_for_tuning * n_train_val)
n_train_tune = n_train_val - n_val_tune
train_tune_end_idx = n_train_tune
val_tune_start_idx = n_train_tune
val_tune_end_idx = n_train_val
train_tune_values_for_scaler = df_full['Close'].values[:train_tune_end_idx].reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_tune_values_for_scaler)
print("Scaler fitted on initial training portion (excluding tuning validation).")
scaled_data = scaler.transform(df_full['Close'].values.reshape(-1, 1))
scaled_train_tune_data = scaled_data[:train_tune_end_idx]


# %% [markdown]
# ## 6. Sequence Generation Function (Modified for t+h Horizon)
# (Keep Section 6 function definition as before)
def create_sequences(data, look_back, forecast_horizon):
    """ Creates sequences for time series forecasting. Target is h steps ahead. """
    X, Y = [], []
    if data.ndim == 1: data = data.reshape(-1, 1)
    if len(data) < look_back + forecast_horizon:
        print(f"Warning: Data length ({len(data)}) insufficient for look_back ({look_back}) + forecast_horizon ({forecast_horizon}). Cannot create sequences.")
        return np.array(X), np.array(Y)
    for i in range(look_back, len(data) - forecast_horizon + 1):
        X.append(data[i - look_back:i, 0])
        Y.append(data[i + forecast_horizon - 1, 0])
    return np.array(X), np.array(Y)

# %% [markdown]
# ## 7. Prepare Data for Tuning
# (Keep Section 7 logic as before, but add the robust check at the end)
print(f"\n--- Preparing Sequences for Tuning (t+{forecast_horizon}) ---")
x_train_tune, y_train_tune = create_sequences(scaled_train_tune_data, look_back, forecast_horizon)
if x_train_tune.shape[0] == 0:
    raise ValueError("Training set for tuning is empty.")
x_train_tune = np.reshape(x_train_tune, (x_train_tune.shape[0], x_train_tune.shape[1], 1))

val_seq_data_start_idx = val_tune_start_idx - look_back
val_seq_data_end_idx = val_tune_end_idx + forecast_horizon - 1
val_seq_data_start_idx = max(0, val_seq_data_start_idx)
val_seq_data_end_idx = min(val_seq_data_end_idx, len(scaled_data))
val_tune_data_for_seq = scaled_data[val_seq_data_start_idx : val_seq_data_end_idx]
x_val_all_possible, y_val_all_possible = create_sequences(val_tune_data_for_seq, look_back, forecast_horizon)

val_input_start_orig_idx = val_tune_start_idx
val_input_end_orig_idx = val_tune_end_idx
first_seq_input_end_idx = val_seq_data_start_idx + look_back - 1
start_j = max(0, val_input_start_orig_idx - first_seq_input_end_idx)
end_j = val_input_end_orig_idx - first_seq_input_end_idx

num_generated_val_seq = x_val_all_possible.shape[0]
start_j = min(start_j, num_generated_val_seq)
end_j = min(end_j, num_generated_val_seq)

if start_j >= end_j:
    print(f"Warning: Calculated validation slice [{start_j}:{end_j}] is invalid or empty.")
    x_val_tune = np.array([])
    y_val_tune = np.array([])
else:
    x_val_tune = x_val_all_possible[start_j:end_j]
    y_val_tune = y_val_all_possible[start_j:end_j]
    x_val_tune = np.reshape(x_val_tune, (x_val_tune.shape[0], x_val_tune.shape[1], 1))

print('x_train_tune shape:', x_train_tune.shape)
print('y_train_tune shape:', y_train_tune.shape)
print('x_val_tune shape:', x_val_tune.shape)
print('y_val_tune shape:', y_val_tune.shape)

# >>> Robust Check before Tuner <<<
if x_val_tune.shape[0] == 0 or y_val_tune.shape[0] == 0:
    raise ValueError("Validation set for tuning is empty *after* alignment slicing. Check data splits, look_back, and forecast_horizon interaction.")
if x_train_tune.shape[0] == 0 or y_train_tune.shape[0] == 0:
     raise ValueError("Training set for tuning is empty.")


# %% [markdown]
# ## 8. LSTM Model Building Function for Keras Tuner
# (Keep Section 8 function `build_model` as before)
def build_model(hp):
    """Builds LSTM model with tunable hyperparameters."""
    model = Sequential(name=f"LSTM_t{forecast_horizon}_Tuner")
    model.add(LSTM(units=hp.Int('units_1', min_value=32, max_value=128, step=32),
                   return_sequences=True,
                   input_shape=(look_back, 1)))
    model.add(Dropout(rate=hp.Float('dropout_1', min_value=0.0, max_value=0.3, step=0.1)))
    model.add(LSTM(units=hp.Int('units_2', min_value=32, max_value=128, step=32),
                   return_sequences=False))
    model.add(Dropout(rate=hp.Float('dropout_2', min_value=0.0, max_value=0.3, step=0.1)))
    model.add(Dense(units=hp.Int('dense_units', min_value=16, max_value=64, step=16), activation='relu'))
    model.add(Dense(1, name='Output'))
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    hp_batch_size = hp.Choice('batch_size', values=[16, 32, 64])
    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss='mean_squared_error')
    return model

# %% [markdown]
# ## 9. Hyperparameter Tuning
# (Keep Section 9 tuner setup and search as before, using the specific directory/project names)
print(f"\n--- Starting Hyperparameter Search with Keras Tuner (t+{forecast_horizon}) ---")
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=MAX_TRIALS,
    executions_per_trial=EXECUTIONS_PER_TRIAL,
    directory=tuner_dir, # Use specific directory per horizon
    project_name=project_name,
    overwrite=False # Set to True to force re-run tuning
)

tuner_early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=TUNER_PATIENCE, verbose=1)

# Check if tuner already has results
search_needed = True
if os.path.exists(os.path.join(tuner_dir, project_name)):
    print("Existing tuning results found. Attempting to load best hyperparameters.")
    try:
        # Try loading HPs first to see if results are usable
        best_hps_check = tuner.get_best_hyperparameters(num_trials=1)[0]
        print("Successfully loaded existing hyperparameters.")
        search_needed = False # Skip search if loading works
    except Exception as load_e:
        print(f"Could not load existing hyperparameters ({load_e}). Re-running search with overwrite=True.")
        # Clean up potentially corrupted directory before overwrite
        import shutil
        try:
            shutil.rmtree(os.path.join(tuner_dir, project_name))
        except OSError as clean_e:
            print(f"Error removing directory during cleanup: {clean_e}")
        tuner = kt.RandomSearch( # Re-initialize with overwrite=True
            build_model,
            objective='val_loss',
            max_trials=MAX_TRIALS,
            executions_per_trial=EXECUTIONS_PER_TRIAL,
            directory=tuner_dir,
            project_name=project_name,
            overwrite=True)

if search_needed:
    print("Starting hyperparameter search...")
    # Use the prepared tuning sequences
    tuner.search(x_train_tune, y_train_tune,
                 epochs=TUNER_EPOCHS,
                 validation_data=(x_val_tune, y_val_tune),
                 callbacks=[tuner_early_stopping],
                 verbose=1) # Show search progress

# Get the optimal hyperparameters
try:
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
except Exception as e:
    print(f"Error retrieving best hyperparameters *after* search: {e}.")
    raise RuntimeError("Hyperparameter tuning failed or results are inaccessible even after running/checking search.") from e


# Retrieve the best batch size from hyperparameters
tuned_batch_size = best_hps.get('batch_size')

print(f"""
--- Hyperparameter Search Complete (t+{forecast_horizon}) ---
Best Hyperparameters Found:
- LSTM Layer 1 Units: {best_hps.get('units_1')}
- Dropout 1 Rate: {best_hps.get('dropout_1'):.2f}
- LSTM Layer 2 Units: {best_hps.get('units_2')}
- Dropout 2 Rate: {best_hps.get('dropout_2'):.2f}
- Dense Layer Units: {best_hps.get('dense_units')}
- Learning Rate: {best_hps.get('learning_rate')}
- Batch Size: {tuned_batch_size}
""")


# %% [markdown]
# ## 10. Train Final Initial Model with Best Hyperparameters
# (Keep Section 10 logic as before, using the specific h=10 horizon)
print(f"\n--- Training Final Initial LSTM Model (t+{forecast_horizon}) ---")
start_time_initial_train = time.time()
train_val_data_end_idx_for_seq = n_train_val + forecast_horizon - 1
train_val_data_end_idx_for_seq = min(train_val_data_end_idx_for_seq, len(scaled_data))
scaled_train_val_data_for_seq = scaled_data[:train_val_data_end_idx_for_seq]
x_train_val_final, y_train_val_final = create_sequences(scaled_train_val_data_for_seq, look_back, forecast_horizon)
num_final_train_sequences = n_train_val - look_back
if num_final_train_sequences <= 0: raise ValueError("Not enough data for final training sequences.")
x_train_val_final = x_train_val_final[:num_final_train_sequences]
y_train_val_final = y_train_val_final[:num_final_train_sequences]
x_train_val_final = np.reshape(x_train_val_final, (x_train_val_final.shape[0], x_train_val_final.shape[1], 1))
print(f"Final training sequences shape: X={x_train_val_final.shape}, Y={y_train_val_final.shape}")
final_lstm_model = tuner.hypermodel.build(best_hps)
final_early_stopping = callbacks.EarlyStopping(monitor='loss', patience=FINAL_TRAINING_PATIENCE, restore_best_weights=True, verbose=1)
print(f"Training final initial LSTM on {x_train_val_final.shape[0]} sequences for up to {FINAL_TRAINING_EPOCHS} epochs...")
history_final = final_lstm_model.fit(
    x_train_val_final, y_train_val_final,
    epochs=FINAL_TRAINING_EPOCHS,
    batch_size=tuned_batch_size,
    callbacks=[final_early_stopping],
    verbose=1
)
end_time_initial_train = time.time()
print(f"Final Initial LSTM Training Complete in {end_time_initial_train - start_time_initial_train:.2f} seconds.")
final_lstm_model.summary()


# %% [markdown]
# ## 11. Walk-Forward Validation (Rolling Forecast) Loop
# (Keep Section 11 logic as before, it's independent of h once model is trained)
print(f"\n--- Starting LSTM Walk-Forward Validation for {n_test} steps (t+{forecast_horizon}) ---")
start_time_walk_forward = time.time()
lstm_walk_forward_predictions = []
history_scaled = scaled_data[:n_train_val].flatten().tolist()
for i in range(n_test):
    current_full_index = n_train_val + i
    if len(history_scaled) < look_back: raise IndexError(f"History length short at step {i}.")
    input_sequence = np.array(history_scaled[-look_back:]).reshape((1, look_back, 1))
    pred_scaled = final_lstm_model.predict(input_sequence, verbose=0)[0, 0]
    pred_unscaled = scaler.inverse_transform([[pred_scaled]])[0, 0]
    lstm_walk_forward_predictions.append(pred_unscaled)
    if current_full_index < len(scaled_data):
        actual_scaled_value_i = scaled_data[current_full_index, 0]
        history_scaled.append(actual_scaled_value_i)
    else: print(f"Reached end of actual data at step {i}.")
    if RETRAIN_FREQUENCY > 0 and (i + 1) % RETRAIN_FREQUENCY == 0 and (i + 1) < n_test : pass # Retraining logic omitted for brevity
    elif (i + 1) % 100 == 0: print(f"LSTM Walk-Forward Step {i+1}/{n_test} complete.")
end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nLSTM Walk-Forward finished in {total_walk_forward_time:.2f} seconds.")
lstm_walk_forward_predictions = np.array(lstm_walk_forward_predictions)
print(f"Generated {len(lstm_walk_forward_predictions)} predictions.")


# %% [markdown]
# ## 12. Evaluate Walk-Forward Performance
# (Keep Section 12 logic as before, using forecast_horizon=10)
def evaluate_forecast(y_true, y_pred, model_name, horizon):
    """Calculates and prints standard evaluation metrics."""
    if len(y_true) == 0 or len(y_pred) == 0:
        print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
        print("Evaluation skipped: No data points to evaluate.")
        return {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}
    y_true_flat=np.array(y_true).flatten(); y_pred_flat=np.array(y_pred).flatten()
    valid_indices=~np.isnan(y_true_flat)&~np.isnan(y_pred_flat)&~np.isinf(y_true_flat)&~np.isinf(y_pred_flat)
    y_true_clean=y_true_flat[valid_indices]; y_pred_clean=y_pred_flat[valid_indices]
    if len(y_true_clean)==0: print(f"\n--- {model_name} Evaluation Skipped: No valid points ---"); return {'RMSE':np.nan,'MAE':np.nan,'MAPE':np.nan,'R2':np.nan}
    mae=mean_absolute_error(y_true_clean, y_pred_clean); mask=y_true_clean!=0
    if np.any(mask): mape=np.mean(np.abs((y_true_clean[mask]-y_pred_clean[mask])/y_true_clean[mask]))
    else: mape=np.nan
    rmse=np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    try: r2=r2_score(y_true_clean, y_pred_clean) if np.var(y_true_clean)>1e-9 else np.nan
    except ValueError: r2=np.nan
    print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---"); print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4%}, R²: {r2:.4f}"); print(f"Number of evaluation points: {len(y_true_clean)}")
    return {'RMSE':rmse,'MAE':mae,'MAPE':mape,'R2':r2}

start_actual_idx = n_train_val + forecast_horizon - 1
end_actual_idx = n_train_val + n_test + forecast_horizon - 1
max_actual_idx = len(df_full); end_actual_idx = min(end_actual_idx, max_actual_idx)
y_test_actual = df_full['Close'].values[start_actual_idx : end_actual_idx]
num_eval_points = len(y_test_actual)
lstm_walk_forward_predictions_eval = lstm_walk_forward_predictions[:num_eval_points]
print(f"\n--- Evaluation Alignment ---"); print(f"Number of predictions made: {len(lstm_walk_forward_predictions)}"); print(f"Actual data start index: {start_actual_idx}, end index (excl): {end_actual_idx}"); print(f"Points for eval: Actual={len(y_test_actual)}, Preds={len(lstm_walk_forward_predictions_eval)}")
if len(y_test_actual) != len(lstm_walk_forward_predictions_eval): raise ValueError("Length mismatch after alignment.")
lstm_wf_results = evaluate_forecast(y_test_actual, lstm_walk_forward_predictions_eval, f"LSTM ({ticker})", forecast_horizon)


# %% [markdown]
# ## 13. Visualize Walk-Forward Results
# (Keep Section 13 logic as before, using forecast_horizon=10)
print("\n--- Plotting Walk-Forward Forecasts ---")
actual_dates = df_full.index[start_actual_idx : end_actual_idx]
if len(actual_dates) != len(lstm_walk_forward_predictions_eval):
     min_plot_len=min(len(actual_dates),len(lstm_walk_forward_predictions_eval)); actual_dates=actual_dates[:min_plot_len]; lstm_walk_forward_predictions_plot=lstm_walk_forward_predictions_eval[:min_plot_len]; y_test_actual_plot=y_test_actual[:min_plot_len]
else: lstm_walk_forward_predictions_plot=lstm_walk_forward_predictions_eval; y_test_actual_plot=y_test_actual
if len(actual_dates) > 0:
    results_df_wf=pd.DataFrame({'Actual':y_test_actual_plot.flatten(), f'LSTM (t+{forecast_horizon})':lstm_walk_forward_predictions_plot.flatten()}, index=actual_dates)
    fig=go.Figure(); fig.add_trace(go.Scatter(x=results_df_wf.index,y=results_df_wf['Actual'],mode='lines',name='Actual Price (Eval Period)',line=dict(color='black'))); fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'LSTM (t+{forecast_horizon})'], mode='lines', name=f'LSTM Walk-Forward (t+{forecast_horizon})', line=dict(color='green', dash='dash')))
    fig.update_layout(title=f'LSTM Walk-Forward (t+{forecast_horizon}) Forecast Comparison for {ticker} (Tuned)',xaxis_title="Date (Target Date of Forecast)",yaxis_title="Price (USD)",legend_title="Data/Model",template="plotly_white"); fig.show()
else: print("No evaluation points to plot.")


# %% [markdown]
# ## 14. Walk-Forward Evaluation Period Summary
# (Keep Section 14 logic as before, using forecast_horizon=10)
print(f"\n--- Walk-Forward Evaluation Summary (t+{forecast_horizon}) ---")
print(f"Initial Train+Validation Data End Date: {train_val_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps Made: {n_test}")
print(f"Number of Forecasts Evaluated: {num_eval_points}")
if num_eval_points > 0 and len(actual_dates) > 0: print(f"Evaluation Period (Target Dates): {results_df_wf.index.min().strftime('%Y-%m-%d')} to {results_df_wf.index.max().strftime('%Y-%m-%d')}")
else: print("Evaluation Period: N/A")
print(f"Forecast Horizon: {forecast_horizon} days")

Trial 10 Complete [00h 00m 44s]
val_loss: 0.0021207607933320105

Best val_loss So Far: 0.00145619927207008
Total elapsed time: 00h 06m 57s

--- Hyperparameter Search Complete (t+10) ---
Best Hyperparameters Found:
- LSTM Layer 1 Units: 96
- Dropout 1 Rate: 0.20
- LSTM Layer 2 Units: 64
- Dropout 2 Rate: 0.10
- Dense Layer Units: 32
- Learning Rate: 0.01
- Batch Size: 64


--- Training Final Initial LSTM Model (t+10) ---
Final training sequences shape: X=(2028, 60, 1), Y=(2028,)
Training final initial LSTM on 2028 sequences for up to 100 epochs...
Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - loss: 0.0719
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - loss: 0.0046
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 0.0034
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - loss: 0.0034
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - loss: 0.0042
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - loss: 0.0035
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 65m

Model: "LSTM_t10_Tuner"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 60, 96)         │        37,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 96)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 242,885 (948.77 KB)

 Trainable params: 80,961 (316.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 161,924 (632.52 KB)


--- Starting LSTM Walk-Forward Validation for 513 steps (t+10) ---
LSTM Walk-Forward Step 100/513 complete.
LSTM Walk-Forward Step 200/513 complete.
LSTM Walk-Forward Step 300/513 complete.
LSTM Walk-Forward Step 400/513 complete.
LSTM Walk-Forward Step 500/513 complete.

LSTM Walk-Forward finished in 22.14 seconds.
Generated 513 predictions.

--- Evaluation Alignment ---
Number of predictions made: 513
Actual data start index: 2097, end index (excl): 2610
Points for eval: Actual=513, Preds=513

--- LSTM (BTC-USD) Walk-Forward (t+10) Evaluation Results ---
RMSE: 6361.2464, MAE: 4367.6504, MAPE: 7.0139%, R²: 0.8964
Number of evaluation points: 513

--- Plotting Walk-Forward Forecasts ---



--- Walk-Forward Evaluation Summary (t+10) ---
Initial Train+Validation Data End Date: 2023-07-28
Number of Walk-Forward Steps Made: 513
Number of Forecasts Evaluated: 513
Evaluation Period (Target Dates): 2023-08-07 to 2024-12-31
Forecast Horizon: 10 days
